# The arrow the data will never draw

Somebody runs a structure-learning package on the warehouse and gets a diagram with arrows on
it. The arrows look like findings. Half of them are the algorithm's tie-break, and re-running
it on next quarter's data would point some of them the other way — not because anything
changed, but because *nothing in observational data distinguishes the two directions*.

Observational data cannot identify a DAG. It identifies a **Markov equivalence
class**: every member entails exactly the same conditional independencies, so
no amount of observation separates them. The honest output of structure
learning is therefore the class — an *essential graph*, directed where every
member agrees and undirected where they do not.

That fact has a practical edge. If you know which edges observation could never
orient, you know which questions need an experiment — and `orientation_gain`
prices a proposed experiment in edges before it is run.

In [ ]:
import numpy as np

from axiom.discover import (
    Dataset, DiscoveryResult, EssentialGraph, GaussianBIC, consistent_extension,
    consistent_extensions, cpdag, ges, gies, interventional_essential_graph,
    markov_equivalent, meek_closure, orientation_gain, v_structures,
)
from axiom.identify import CausalGraph

from axiom.display import enable, table
from axiom.viz import causal_graph

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, lines

enable();  # every axiom result renders itself from here on

truth = CausalGraph.from_edges("a -> b, b -> c, a -> d, d -> c")
print("the world:", truth.to_text())

## The class, not the graph

A chain `a -> b -> c` and its reverse entail the same independencies, so
observation cannot tell them apart; a collider `a -> c <- b` entails a
*different* one (a and b are marginally independent, dependent given c) and is
therefore fully oriented by observation alone. That asymmetry — v-structures —
is the only thing observational data can orient directly. Meek's rules
propagate the rest.

In [ ]:
print("chain    :", cpdag(CausalGraph.from_edges("a -> b, b -> c")).to_text())
print("collider :", cpdag(CausalGraph.from_edges("a -> c, b -> c")).to_text())
print("v-structures of the collider:", v_structures(CausalGraph.from_edges("a -> c, b -> c")))
print()
essential: EssentialGraph = cpdag(truth)
print("our world's CPDAG:", essential.to_text())
print(f"  {essential.oriented} oriented, {essential.undecided} left undecided")

In [ ]:
causal_graph(truth, height=340)

In [ ]:
# Every DAG in the class -- what observation genuinely cannot distinguish.
table(
    [[member.to_text()] for member in consistent_extensions(essential)],
    headers=("DAG consistent with the class",),
)
print("all Markov equivalent to the truth:",
      all(markov_equivalent(m, truth) for m in consistent_extensions(essential)))
print("a representative:", consistent_extension(essential).to_text())

## Pricing an experiment before running it

Randomizing a variable severs its incoming edges, so an edge with **exactly
one** endpoint in the target has its direction revealed. Meek's rules then
propagate whatever that forces. Intervening on *both* ends of an edge reveals
nothing about it — both were randomized — which is why the biggest experiment
is not the most informative one.

In [ ]:
rows = []
for target in (["a"], ["b"], ["c"], ["d"], ["a", "b"]):
    gain = orientation_gain(truth, [target])
    rows.append([str(target), str(gain) if gain else "(nothing new)"])
table(rows, headers=("intervene on", "orients"))

In [ ]:
candidates = [["a"], ["b"], ["c"], ["d"], ["a", "b"]]
bought = {", ".join(t): len(orientation_gain(truth, [t])) for t in candidates}
fig = compare(
    [f"randomize {k}" for k in bought], list(bought.values()),
    highlight="randomize a",
    value_fmt="{:.0f}",
    title="What each experiment would buy, in edges",
    subtitle="edges the intervention orients that observation could not",
    x_title="edges newly oriented",
)
caption(fig, "Randomizing a orients two edges. Randomizing a *and* b — a larger, more "
             "expensive experiment — orients one, because both ends of a→b are then inside "
             "the target and the comparison that would have settled it is gone. Randomizing c "
             "buys nothing at all: every edge touching it was already oriented by a collider.")

In [ ]:
after = interventional_essential_graph(truth, [["a"]])
print("after do(a):", after.to_text(), "| fully determined:", after.is_dag)
# Meek propagation in action: orienting a - b forces nothing more here, but
# closing the graph by hand shows the rules are what finish the job.
partial = EssentialGraph(nodes=truth.nodes, directed=(("a", "b"),), undirected=(("b", "c"),))
print("R1 propagation:", meek_closure(partial).to_text())

## Learning the class from data: GES

`ges` is greedy equivalence search (Chickering 2002): add edges while the score
improves, then remove, then turn, repeating until nothing helps. The score is a
decomposable BIC, and it is *score-equivalent* on observational data — every
member of a class scores identically, which is exactly why the search should
work in class space.

In [ ]:
coefficients = {("a", "b"): 1.4, ("b", "c"): -0.9, ("a", "d"): 0.8, ("d", "c"): 1.1}


def simulate(n, do=(), seed=0):
    rng = np.random.default_rng(seed)
    order = list(truth.topological_order())
    values = {}
    for node in order:
        if node in do:
            values[node] = rng.normal(0.0, 1.5, n)   # randomized: parents disconnected
            continue
        column = rng.normal(0.0, 1.0, n)
        for parent in sorted(truth.parents(node)):
            column = column + coefficients[(parent, node)] * values[parent]
        values[node] = column
    return Dataset(np.column_stack([values[n] for n in order]), tuple(order), (frozenset(do),) * n)


observational = simulate(4000, seed=1)
found: DiscoveryResult = ges(GaussianBIC(observational))
print("GES found  :", found.essential.to_text())
print("true CPDAG :", cpdag(truth).to_text())
print(found.summary())

## Adding experiments: GIES

`gies` is the same search over *interventional* essential graphs, scored with
data that knows which rows were randomized. A row where `b` was randomized
carries no information about what causes `b` — and full information about
everything else, which is why an interventional row is worth more than its
count suggests.

In [ ]:
pooled = Dataset.stack(observational, simulate(2000, do=("b",), seed=2))
print("targets in the data:", pooled.targets)
one = gies(GaussianBIC(pooled))
print("GIES after do(b) :", one.essential.to_text())
print("theory said       :", interventional_essential_graph(truth, [["b"]]).to_text())

In [ ]:
everything = Dataset.stack(
    pooled, simulate(2000, do=("a",), seed=3), simulate(2000, do=("d",), seed=4)
)
full = gies(GaussianBIC(everything))
print("GIES after do(a), do(b), do(d):", full.essential.to_text())
print("is the DAG now determined?", full.essential.is_dag)
print("does it match the world?", set(full.essential.directed) == set(truth.edges))
print()
print("what the search did:")
table([[step] for step in full.steps], headers=("search step",))

The **turning** phase in that log is not decoration. An interventional
essential graph orients a cut edge the moment the edge is inserted, so a
forward step that guesses a direction wrongly is locked in — no insert or
delete escapes it. Without turning, this search stops at the complete graph,
about 1,700 nats below the truth.

Two cautions the module states rather than assumes. Every method here assumes
**faithfulness**, which is not testable from the data it is assumed about. And
greedy is greedy: the penalty is a dial, but not a monotone one — raising it
changes the *path* as well as the destination, and can land on a graph with
more edges than a lower penalty found.

In [ ]:
rows = []
for penalty in (1.0, 20.0, 400.0):
    result = ges(GaussianBIC(observational, penalty=penalty))
    rows.append([penalty, len(result.essential.skeleton), result.essential.to_text() or "(empty)"])
table(rows, headers=("penalty", "edges", "essential graph"))

In [ ]:
penalties = (0.5, 1.0, 2.0, 5.0, 20.0, 100.0, 400.0)
edge_counts = [len(ges(GaussianBIC(observational, penalty=pen)).essential.skeleton) for pen in penalties]
fig = lines(
    np.log10(penalties), {"edges found": edge_counts},
    title="The penalty is a dial, and it is not a monotone one",
    subtitle="edges in the learned class as the BIC penalty rises",
    x_title="log₁₀ penalty", y_title="edges",
)
caption(fig, f"The truth has {len(truth.edges)} edges. Raising the penalty changes the "
             f"search *path* as well as the destination, so a higher penalty can land on a "
             f"denser graph than a lower one did — which is why the number is reported rather "
             f"than tuned quietly.")
print()
print("limits reported:", full.limits_hit or "(none)")

## A discovered graph is a hypothesis

The rest of `axiom` treats it as one. The natural next step is to hand the
class to identification and ask what it would license — and to notice that
different members of the class may disagree, which is itself worth reporting.

In [ ]:
from axiom.identify import identify_effect

table(
    [[m.to_text(), identify_effect(m, "a", "c").to_text()] for m in consistent_extensions(cpdag(truth))],
    headers=("member of the class", "identifies a -> c as"),
)

## What this bought you

An answer that distinguishes "the data says a causes b" from "the data says a and b are
adjacent and cannot say more" — and, for the second kind, the experiment that would settle
it, priced in edges before anything is randomized.

`02-stability-and-hidden-causes.ipynb` asks how much of a single run's answer survives
resampling, and what to do when the thing driving everything was never measured.